# Week5_ex1 — Balanced three-phase faults on a five-bus system

Reproduces Example 8.5 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a bolted three-phase fault at each bus, solved with the bus impedance matrix and checked against PowerWorld's fault analysis (pre-fault profile = solved power flow, all buses at 1.05 pu).

![PowerWorld one-line](Images/Week5_ex1_result_image.png)

In [1]:
import numpy as np
import pandas as pd
import pandapower as pp
from case5 import (S_BASE, V_PREFAULT, BUSES, build_case, conversion_table,
                   ybus_with_generators, classical_fault, compare)

SRC = "PowerWorld Fault Analysis (Run Mode)"

# 1. Per-unit to pandapower conversion (Z_base = V_kV^2 / S_base, transformers on S_base)
conversion_table().round(4)

,Branch,X [pu],Z_base [ohm],Input,Value
0,T 5-1,0.020,NaN,vk_percent,2.0000
1,T 4-3,0.010,NaN,vk_percent,1.0000
2,L 4-2,0.100,1190.25,x_ohm_per_km (length_km=1),119.0250
3,L 5-2,0.050,1190.25,x_ohm_per_km (length_km=1),59.5125
4,L 5-4,0.025,1190.25,x_ohm_per_km (length_km=1),29.7562


In [2]:
# 2. Pre-fault state: no load, both generators hold 1.05 pu
net = build_case()
pp.runpp(net, numba=False)
net.res_bus[["vm_pu", "va_degree"]].rename(
    columns={"vm_pu": "Voltage [pu]", "va_degree": "Angle [deg]"}).round(5)

,Voltage [pu],Angle [deg]
1,1.05,0.0
2,1.05,0.0
3,1.05,0.0
4,1.05,0.0
5,1.05,0.0


In [3]:
# 3. Positive-sequence Ybus with generator subtransient admittances
Y = ybus_with_generators(net)
Z = np.linalg.inv(Y)

# 4. PowerWorld Fault Analysis > Y-Bus Matrices > Positive Sequence (imaginary parts)
y_pw = {(1, 1): -72.22, (1, 5): 50.00, (2, 2): -30.00, (2, 4): 10.00, (2, 5): 20.00,
        (3, 3): -144.44, (3, 4): 100.00, (4, 4): -150.00, (4, 5): 40.00, (5, 5): -110.00}

# 5. Two displayed decimals -> half a digit tolerance
rows = [(f"B{i}{j} [pu]", v, Y[BUSES.index(i), BUSES.index(j)].imag, 5e-3, SRC)
        for (i, j), v in y_pw.items()]
compare(rows).round(4)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,B11 [pu],-72.22,-72.2222,0.0022,0.005,PowerWorld Fault Analysis (Run Mode)
1,B15 [pu],50.00,50.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
2,B22 [pu],-30.00,-30.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
3,B24 [pu],10.00,10.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
4,B25 [pu],20.00,20.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
5,B33 [pu],-144.44,-144.4444,0.0044,0.005,PowerWorld Fault Analysis (Run Mode)
6,B34 [pu],100.00,100.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
7,B44 [pu],-150.00,-150.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
8,B45 [pu],40.00,40.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)
9,B55 [pu],-110.00,-110.0000,0.0000,0.005,PowerWorld Fault Analysis (Run Mode)


In [4]:
# 6. Fault at every bus: Thevenin reactance (Zbus diagonal) and fault current
i_pw = {1: 37.53623, 2: 18.43659, 3: 57.55555, 4: 44.45645, 5: 35.62404}
x_pw = {1: 0.02797, 2: 0.05695, 3: 0.01824, 4: 0.02362, 5: 0.02947}

# 7. Currents use one displayed digit (1e-5): PowerWorld's pre-fault voltage is a solved
#    value just under 1.05, which moves the fifth decimal of the largest current
rows = []
for b in BUSES:
    k = BUSES.index(b)
    i_f, _ = classical_fault(Z, k)
    rows.append((f"Bus {b} Thev X [pu]", x_pw[b], abs(Z[k, k]), 5e-6, SRC))
    rows.append((f"Bus {b} I_F [pu]", i_pw[b], abs(i_f), 1e-5, SRC))
compare(rows).round(6)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,Bus 1 Thev X [pu],0.02797,0.027973,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
1,Bus 1 I_F [pu],37.53623,37.536232,0.000002,0.000010,PowerWorld Fault Analysis (Run Mode)
2,Bus 2 Thev X [pu],0.05695,0.056952,0.000002,0.000005,PowerWorld Fault Analysis (Run Mode)
3,Bus 2 I_F [pu],18.43659,18.436594,0.000004,0.000010,PowerWorld Fault Analysis (Run Mode)
4,Bus 3 Thev X [pu],0.01824,0.018243,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
5,Bus 3 I_F [pu],57.55555,57.555556,0.000006,0.000010,PowerWorld Fault Analysis (Run Mode)
6,Bus 4 Thev X [pu],0.02362,0.023619,0.000001,0.000005,PowerWorld Fault Analysis (Run Mode)
7,Bus 4 I_F [pu],44.45645,44.456453,0.000003,0.000010,PowerWorld Fault Analysis (Run Mode)
8,Bus 5 Thev X [pu],0.02947,0.029474,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
9,Bus 5 I_F [pu],35.62404,35.624045,0.000005,0.000010,PowerWorld Fault Analysis (Run Mode)


In [5]:
# 8. Fault at bus 1: voltages during the fault
i_f, V = classical_fault(Z, BUSES.index(1))
v_pw = {1: 0.0, 2: 0.38551, 3: 0.73043, 4: 0.58841, 5: 0.28406}
rows = [(f"Bus {b} V [pu]", v_pw[b], abs(V[BUSES.index(b)]), 5e-6, SRC) for b in BUSES]
compare(rows).round(6)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,Bus 1 V [pu],0.00000,0.000000,0.000000,0.000005,PowerWorld Fault Analysis (Run Mode)
1,Bus 2 V [pu],0.38551,0.385507,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
2,Bus 3 V [pu],0.73043,0.730435,0.000005,0.000005,PowerWorld Fault Analysis (Run Mode)
3,Bus 4 V [pu],0.58841,0.588406,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
4,Bus 5 V [pu],0.28406,0.284058,0.000002,0.000005,PowerWorld Fault Analysis (Run Mode)


In [6]:
# 9. Fault at bus 1: branch and generator contributions, I = (V_from - V_to) / jX
v = dict(zip(BUSES, V))
branches = {"T 5-1": (5, 1, 0.02), "T 4-3": (4, 3, 0.01), "L 4-2": (4, 2, 0.10),
            "L 5-2": (5, 2, 0.05), "L 5-4": (5, 4, 0.025)}
i_br_pw = {"T 5-1": 14.20290, "T 4-3": 14.20290, "L 4-2": 2.02899, "L 5-2": 2.02899, "L 5-4": 12.17391}
rows = [(f"{n} I [pu]", i_br_pw[n], abs((v[f] - v[t]) / (1j * x)), 5e-6, SRC)
        for n, (f, t, x) in branches.items()]

# 10. Generator current flows from the internal 1.05 pu source through X''d
rows += [("G1 I [pu]", 23.33333, abs((V_PREFAULT - v[1]) / 0.045j), 5e-6, SRC),
         ("G3 I [pu]", 14.20290, abs((V_PREFAULT - v[3]) / 0.0225j), 5e-6, SRC)]
compare(rows).round(6)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,T 5-1 I [pu],14.20290,14.202899,0.000001,0.000005,PowerWorld Fault Analysis (Run Mode)
1,T 4-3 I [pu],14.20290,14.202899,0.000001,0.000005,PowerWorld Fault Analysis (Run Mode)
2,L 4-2 I [pu],2.02899,2.028986,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
3,L 5-2 I [pu],2.02899,2.028986,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
4,L 5-4 I [pu],12.17391,12.173913,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
5,G1 I [pu],23.33333,23.333333,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
6,G3 I [pu],14.20290,14.202899,0.000001,0.000005,PowerWorld Fault Analysis (Run Mode)
